# Exploratory Data Analysis — ALPR Dataset Pipeline

This notebook walks through Part 1 (Inspection) and Part 2 (EDA) of the pipeline.
It scans the raw datasets, computes per-image statistics, detects duplicates,
and generates publication-quality figures saved to `reports/figures/`.

**Outputs:**
- `reports/figures/<dataset_name>/*.png/.svg` — 22 figure types per dataset
- `reports/figures/dataset_size_comparison.png/.svg` — cross-dataset bar chart
- `reports/figures/<dataset_name>_eda_report.md` — per-dataset summary report
- `reports/eda/dataset_summary.md/.csv` — inspection summary

**Requirements:** `pip install -r requirements.txt`

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# --- path setup ---
# Walk up to find project root (handles kernels with different CWD)
_candidate = Path.cwd().resolve()
for _parent in [_candidate] + list(_candidate.parents):
    if (_parent / "src" / "alpr_dataset").is_dir():
        PROJECT_ROOT = _parent
        break
else:
    PROJECT_ROOT = _candidate
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

---
## 1. Load Configuration

In [ ]:
from alpr_dataset.config import PipelineConfig
from alpr_dataset.logging_setup import setup_logging
from alpr_dataset.io_utils import list_images

config = PipelineConfig.load(
    PROJECT_ROOT / "configs" / "pipeline_config.yaml",
    PROJECT_ROOT / "configs" / "datasets.yaml",
)
logger = setup_logging(config.logs_dir, name="alpr_dataset")

print(f"Datasets configured: {[s.name for s in config.datasets]}")
print(f"Reports dir: {config.reports_dir}")
print(f"Blur threshold: {config.blur_threshold}")
print(f"Duplicate hash threshold: {config.duplicate_hash_threshold}")

---
## 2. Part 1 — Dataset Inspection

Scan folder structure, count files, detect formats, check for orphans.

In [ ]:
from alpr_dataset.inspection.scanner import scan_dataset
from alpr_dataset.inspection.report import generate_dataset_summary

dataset_specs = [(s.name, s.root) for s in config.datasets]
df_summary = generate_dataset_summary(
    dataset_specs,
    output_dir=config.reports_dir / "eda",
    hamming_threshold=config.duplicate_hash_threshold,
)
df_summary

In [ ]:
# Per-dataset folder tree
for spec in config.datasets:
    scan = scan_dataset(spec.name, spec.root)
    print(f"\n{'='*60}\n{spec.name}\n{'='*60}")
    print(f"Images: {scan.n_images}  |  Annotations: {scan.n_annotations}")
    print(f"Image formats: {dict(scan.image_format_counts)}")
    print(f"Annotation formats: {dict(scan.annotation_format_counts)}")
    print(f"Missing annotations: {len(scan.images_missing_annotations)}")
    print(f"Orphan annotations: {len(scan.orphan_annotations)}")

---
## 3. Load Annotations

Parses each dataset's annotations into the unified `ImageAnnotation` schema.

In [ ]:
from tqdm import tqdm
from alpr_dataset.annotations.loader import load_dataset_annotations

all_annotations = {}
for spec in config.datasets:
    annots = load_dataset_annotations(spec)
    all_annotations[spec.name] = annots
    n_boxes = sum(a.n_boxes for a in annots)
    print(f"{spec.name}: {len(annots)} images annotated, {n_boxes} bounding boxes total")

In [ ]:
# Quick peek at annotation structure
for name, annots in all_annotations.items():
    if annots:
        a = annots[0]
        print(f"\n{name} — first annotation:")
        print(f"  Image: {a.image_path.name} ({a.image_width}x{a.image_height})")
        print(f"  Boxes: {a.n_boxes}")
        if a.boxes:
            b = a.boxes[0]
            print(f"  First box: class={b.class_name} [{b.x_min:.0f},{b.y_min:.0f}]->[{b.x_max:.0f},{b.y_max:.0f}]")
            print(f"  Box width={b.width:.0f}, height={b.height:.0f}, area={b.area:.0f}")

---
## 4. Compute Image Statistics

Per-image: resolution, brightness, contrast, blur, sharpness, entropy.

In [ ]:
from alpr_dataset.inspection.image_stats import batch_compute_stats
from alpr_dataset.inspection.hashing import find_duplicates

all_data = {}
for spec in config.datasets:
    images = list_images(spec.root)
    stats = batch_compute_stats(images)
    dupes = find_duplicates(images, hamming_threshold=config.duplicate_hash_threshold).near_duplicates
    all_data[spec.name] = {"images": images, "stats": stats, "dupes": dupes}
    valid = [s for s in stats if not s.is_corrupted]
    print(f"\n{spec.name} — {len(images)} images, {len(valid)} valid, {sum(1 for s in stats if s.is_corrupted)} corrupted")
    print(f"  Widths:  {min(s.width for s in valid)}-{max(s.width for s in valid)} px")
    print(f"  Heights: {min(s.height for s in valid)}-{max(s.height for s in valid)} px")
    print(f"  Brightness: {min(s.brightness_mean for s in valid):.1f} / {sum(s.brightness_mean for s in valid)/len(valid):.1f} / {max(s.brightness_mean for s in valid):.1f}")
    print(f"  Blur (VoF): {min(s.blur_score for s in valid):.1f} / {sum(s.blur_score for s in valid)/len(valid):.1f} / {max(s.blur_score for s in valid):.1f}")
    print(f"  Entropy:    {min(s.entropy for s in valid):.2f} / {sum(s.entropy for s in valid)/len(valid):.2f} / {max(s.entropy for s in valid):.2f}")
    print(f"  Near-duplicate pairs: {len(dupes)}")

---
## 5. Part 2 — Generate EDA Figures

All figures are saved as **PNG + SVG** pairs into `reports/figures/<dataset_name>/`.
Inline previews are shown for key figures below.

In [ ]:
from alpr_dataset.eda.figures import (
    plot_dataset_size_comparison,
    plot_resolution_histograms,
    plot_width_distribution,
    plot_height_distribution,
    plot_aspect_ratio_distribution,
    plot_brightness_histogram,
    plot_contrast_histogram,
    plot_blur_estimation,
    plot_sharpness,
    plot_entropy,
    plot_bbox_width_distribution,
    plot_bbox_height_distribution,
    plot_bbox_area_distribution,
    plot_bbox_position_heatmap,
    plot_class_distribution,
    plot_example_images,
    plot_random_samples,
    plot_annotated_samples,
    plot_color_distribution,
    plot_duplicate_visualization,
    plot_outlier_visualization,
    generate_all_eda_figures,
)
from alpr_dataset.eda.report import generate_eda_report

shared_fig_dir = config.reports_dir / "figures"

In [ ]:
# Cross-dataset comparison
dataset_counts = {name: len(d["images"]) for name, d in all_data.items()}
plot_dataset_size_comparison(dataset_counts, shared_fig_dir)
print(f"Cross-dataset comparison saved to {shared_fig_dir}")

In [ ]:
# Generate all figures for each dataset
for spec in config.datasets:
    d = all_data[spec.name]
    output_dir = shared_fig_dir / spec.name
    
    generate_all_eda_figures(
        dataset_counts={spec.name: len(d["images"])},
        stats=d["stats"],
        annotations=all_annotations[spec.name],
        image_paths=d["images"],
        duplicate_pairs=d["dupes"],
        output_dir=output_dir,
        blur_threshold=config.blur_threshold,
    )
    
    generate_eda_report(
        dataset_name=spec.name,
        stats=d["stats"],
        annotations=all_annotations[spec.name],
        image_paths=d["images"],
        duplicate_pairs=d["dupes"],
        figures_dir=output_dir,
        output_dir=shared_fig_dir,
    )
    print(f"{spec.name}: {len(d['images'])} images, figures in {output_dir}")

---
## 6. Inline Figure Previews

Key figures displayed inline for quick visual inspection.

### 6.1 Resolution Distribution

The distribution of image widths and heights across the dataset.

In [ ]:
# Pick the first dataset for inline preview
spec = config.datasets[0]
d = all_data[spec.name]
valid = [s for s in d["stats"] if not s.is_corrupted]
widths = [s.width for s in valid]
heights = [s.height for s in valid]
ratios = [w/h for w,h in zip(widths, heights) if h>0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=40, color="#3d5a80", edgecolor="white")
axes[0].set_xlabel("Width (px)"); axes[0].set_ylabel("Frequency"); axes[0].set_title(f"{spec.name}: Width Distribution")
axes[1].hist(heights, bins=40, color="#ee6c4d", edgecolor="white")
axes[1].set_xlabel("Height (px)"); axes[1].set_ylabel("Frequency"); axes[1].set_title(f"{spec.name}: Height Distribution")
fig.tight_layout(); plt.show()

### 6.2 Aspect Ratio Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratios, bins=40, color="#98c1d9", edgecolor="white")
ax.set_xlabel("Aspect ratio (width/height)")
ax.set_ylabel("Frequency")
ax.set_title(f"{spec.name}: Aspect Ratio Distribution")
fig.tight_layout(); plt.show()

### 6.3 Photometric Quality

Brightness, contrast, blur estimation, sharpness, and entropy histograms.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

brightness = [s.brightness_mean for s in valid]
contrast = [s.contrast_std for s in valid]
blur = [s.blur_score for s in valid]
sharp = [s.sharpness_score for s in valid]
entropy = [s.entropy for s in valid]

axes[0,0].hist(brightness, bins=40, color="#f4d35e", edgecolor="white")
axes[0,0].set_title("Brightness"); axes[0,0].set_xlabel("Mean intensity")
axes[0,1].hist(contrast, bins=40, color="#ee6c4d", edgecolor="white")
axes[0,1].set_title("Contrast"); axes[0,1].set_xlabel("Std dev")
axes[0,2].hist(blur, bins=40, color="#3d5a80", edgecolor="white")
axes[0,2].axvline(100, color="red", linestyle="--", alpha=0.7)
axes[0,2].set_title("Blur (VoF)"); axes[0,2].set_xlabel("Variance of Laplacian")
axes[1,0].hist(sharp, bins=40, color="#98c1d9", edgecolor="white")
axes[1,0].set_title("Sharpness"); axes[1,0].set_xlabel("Sobel magnitude")
axes[1,1].hist(entropy, bins=40, color="#293241", edgecolor="white")
axes[1,1].set_title("Entropy"); axes[1,1].set_xlabel("Shannon entropy (bits)")
axes[1,2].axis("off")
fig.suptitle(f"{spec.name}: Photometric Quality", fontsize=14)
fig.tight_layout(); plt.show()

### 6.4 Bounding Box Analysis

Width, height, area distributions and position heatmap.

In [ ]:
annots = all_annotations[spec.name]
bws = [b.width for a in annots for b in a.boxes if b.width>0]
bhs = [b.height for a in annots for b in a.boxes if b.height>0]
bas = [b.area for a in annots for b in a.boxes if b.area>0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(bws, bins=40, color="#3d5a80", edgecolor="white")
axes[0].set_title("BBox Width"); axes[0].set_xlabel("Width (px)")
axes[1].hist(bhs, bins=40, color="#ee6c4d", edgecolor="white")
axes[1].set_title("BBox Height"); axes[1].set_xlabel("Height (px)")
axes[2].hist(bas, bins=40, color="#4ba36f", edgecolor="white")
axes[2].set_title("BBox Area"); axes[2].set_xlabel("Area (px²)")
fig.suptitle(f"{spec.name}: Bounding Box Size Distributions", fontsize=13)
fig.tight_layout(); plt.show()

In [ ]:
# BBox Position Heatmap
import numpy as np
heat = np.zeros((32, 32), dtype=np.float64)
for ann in annots:
    if ann.image_width <= 0 or ann.image_height <= 0:
        continue
    for box in ann.boxes:
        cx, cy = box.center
        gx = min(int(cx / ann.image_width * 32), 31)
        gy = min(int(cy / ann.image_height * 32), 31)
        heat[gy, gx] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(heat, cmap="inferno", origin="upper")
ax.set_title(f"{spec.name}: BBox Position Heatmap")
plt.colorbar(im, ax=ax, label="Box centre count")
fig.tight_layout(); plt.show()

### 6.5 Sample Images

In [ ]:
import random
import cv2
from alpr_dataset.io_utils import safe_read_image
from alpr_dataset.utils.viz_utils import bgr_to_rgb, draw_boxes

# Show random annotated samples
rng = random.Random(42)
with_boxes = [a for a in annots if a.boxes]
sample = rng.sample(with_boxes, min(6, len(with_boxes)))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for idx, (ann, ax) in enumerate(zip(sample, axes.flat)):
    img = safe_read_image(ann.image_path)
    if img is None:
        continue
    img_rgb = bgr_to_rgb(img)
    for box in ann.boxes:
        import matplotlib.patches as patches
        rect = patches.Rectangle(
            (box.x_min, box.y_min), box.width, box.height,
            fill=False, edgecolor="#ee6c4d", linewidth=2
        )
        ax.add_patch(rect)
    ax.imshow(img_rgb)
    ax.set_title(ann.image_path.name, fontsize=8)
    ax.axis("off")
fig.suptitle(f"{spec.name}: Annotated Samples", fontsize=14)
fig.tight_layout(); plt.show()

### 6.6 Outlier Detection

Scatter plot of resolution vs file size with z-score outliers highlighted.

In [ ]:
resolutions = np.array([s.width * s.height for s in valid])
sizes = np.array([s.file_size_bytes for s in valid])

z_res = (resolutions - resolutions.mean()) / (resolutions.std() or 1)
z_size = (sizes - sizes.mean()) / (sizes.std() or 1)
outlier = (np.abs(z_res) > 3) | (np.abs(z_size) > 3)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(resolutions[~outlier], sizes[~outlier], s=15, alpha=0.5, color="#3d5a80", label="Normal")
ax.scatter(resolutions[outlier], sizes[outlier], s=30, color="#ee6c4d", edgecolor="black", linewidth=0.5, label=f"Outlier ({outlier.sum()})")
ax.set_xlabel("Resolution (W×H, px)"); ax.set_ylabel("File size (bytes)")
ax.set_title(f"{spec.name}: Outlier Detection"); ax.legend()
fig.tight_layout(); plt.show()

---
## 7. Validate Outputs

Check that every expected figure file exists as both PNG and SVG.

In [ ]:
EXPECTED_FIGURES = [
    "resolution_histograms", "width_distribution", "height_distribution",
    "aspect_ratio_distribution", "brightness_histogram", "contrast_histogram",
    "blur_estimation", "sharpness_distribution", "entropy_distribution",
    "bbox_size_distribution", "bbox_width_distribution", "bbox_height_distribution",
    "bbox_area_distribution", "bbox_position_heatmap", "class_distribution",
    "example_images", "random_samples", "annotated_samples", "color_distribution",
    "duplicate_visualization", "outlier_visualization",
]

missing = []
for spec in config.datasets:
    fig_dir = shared_fig_dir / spec.name
    for stem in EXPECTED_FIGURES:
        if not (fig_dir / f"{stem}.png").is_file():
            missing.append(f"{spec.name}/{stem}.png")
        if not (fig_dir / f"{stem}.svg").is_file():
            missing.append(f"{spec.name}/{stem}.svg")

cd_fig = shared_fig_dir / "dataset_size_comparison"
if not (shared_fig_dir / "dataset_size_comparison.png").is_file():
    missing.append("dataset_size_comparison.png")

if missing:
    print(f"MISSING ({len(missing)}):")
    for m in missing:
        print(f"  - {m}")
else:
    total = len(config.datasets) * len(EXPECTED_FIGURES) * 2 + 2
    print(f"✅ All {total} figure files validated (PNG + SVG)")

---
## 8. Summary

| Dataset | Images | Valid | Corrupted | BBoxes | Near-Dupes |
|---------|--------|-------|-----------|--------|------------|
{% for spec in config.datasets -%}
{% set d = all_data[spec.name] -%}
{% set a = all_annotations[spec.name] -%}
| {{ spec.name }} | {{ d['images']|length }} | {{ d['stats']|selectattr('is_corrupted','equalto',False)|list|length }} | {{ d['stats']|selectattr('is_corrupted')|list|length }} | {{ a|sum(attribute='n_boxes') }} | {{ d['dupes']|length }} |
{% endfor %}

All figures saved to `reports/figures/`. EDA reports saved to `reports/figures/*_eda_report.md`.

In [ ]:
# Print summary table
print(f"{'Dataset':<12} {'Images':>8} {'Valid':>8} {'Corrupt':>8} {'BBoxes':>8} {'NearDupes':>10}")
print("-" * 56)
for spec in config.datasets:
    d = all_data[spec.name]
    a = all_annotations[spec.name]
    n_valid = sum(1 for s in d["stats"] if not s.is_corrupted)
    n_corrupt = sum(1 for s in d["stats"] if s.is_corrupted)
    n_boxes = sum(aa.n_boxes for aa in a)
    n_dupes = len(d["dupes"])
    print(f"{spec.name:<12} {len(d['images']):>8} {n_valid:>8} {n_corrupt:>8} {n_boxes:>8} {n_dupes:>10}")

print(f"\n✅ EDA complete. Figures: {shared_fig_dir}")